<a href="https://colab.research.google.com/github/SohamDutta02/paged-kv-cache/blob/real-model/notebooks/qwen_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen2.5 through paged-kv-cache

Downloads Qwen2.5-0.5B-Instruct and answers four chat prompts that share a system prompt. Every layer's K/V lives in paged-kv-cache's block cache: the scheduler decides which blocks each token goes in, paged attention reads them back, and the later prompts reuse the system prompt's blocks from the prefix cache. The same prompts then run through candle's own Qwen2 implementation, and every generated token must match.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. A CPU runtime works too; it just skips the run with the cache on the GPU.
2. The first run takes several minutes, most of it compiling.

In [1]:
# Set BRANCH to test a pull request before it is merged.
REPO = "https://github.com/SohamDutta02/paged-kv-cache"
BRANCH = "real-model"
MAX_NEW_TOKENS = 64

In [2]:
import glob, os, shutil, subprocess

!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":/usr/local/cuda/bin:" + os.environ["PATH"]

HAS_GPU = shutil.which("nvidia-smi") is not None
if HAS_GPU:
    # cudarc loads NVRTC at runtime. Use the toolkit's copy if there is one, else install it.
    nvrtc = sorted(glob.glob("/usr/local/cuda*/lib64/libnvrtc.so*"))
    if nvrtc:
        lib = os.path.dirname(nvrtc[0])
    else:
        subprocess.run(["pip", "install", "-q", "nvidia-cuda-nvrtc-cu12"], check=True)
        import nvidia.cuda_nvrtc
        lib = os.path.join(nvidia.cuda_nvrtc.__path__[0], "lib")
        if not os.path.exists(os.path.join(lib, "libnvrtc.so")):
            os.symlink("libnvrtc.so.12", os.path.join(lib, "libnvrtc.so"))
    os.environ["LD_LIBRARY_PATH"] = lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("GPU runtime:", HAS_GPU)


  stable-x86_64-unknown-linux-gnu installed - rustc 1.99.0 (b940084d7 2026-09-28)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdksh/bash
cargo:rerun-if-env-changed=CC_x86_64-unknown-linux-gnu
CC_x86_64-unknown-linux-gnu = None
cargo:rerun-if-env-changed=CC_x86_64_unknown_linux_gnu
CC_x86_64_unknown_linux_gnu = None
cargo:rerun-if-env-changed=HOST_CC
HOST_CC = None
cargo:rerun-if-env-changed=CC
CC = None
cargo:rerun-if-env-changed=CC_ENABLE_DEBUG_OUTPUT
cargo:rerun-if-env-changed=CRATE_CC_NO_DEFAULTS
CRATE_CC_NO_DEFAULTS = None
cargo:rerun-if-env-changed=CFLAGS
CFLAGS = None
cargo:rerun-if-env-changed=HOST_CFLAGS
HOST_CFLAGS = N

In [3]:
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download(
    "Qwen/Qwen2.5-0.5B-Instruct",
    allow_patterns=["config.json", "tokenizer.json", "model.safetensors"],
)
print(MODEL_DIR)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775


In [4]:
!rm -rf paged-kv-cache && git clone -q --depth 1 --branch {BRANCH} {REPO}
!cd paged-kv-cache && git log -1 --format='Testing %h: %s'

Testing ba5c817: Add a notebook that runs Qwen2.5 through the cache on Colab


In [5]:
# K/V cache in host memory.
!cd paged-kv-cache && cargo run --release -q -p paged-kv-model --example qwen -- {MODEL_DIR} {MAX_NEW_TOKENS}

K/V cache on cpu (host memory reference backend)

Q: What does a GPU do?
A: A GPU (Graphics Processing Unit) is a specialized computer component designed to perform complex mathematical calculations and image processing tasks, enabling it to handle graphics-intensive applications such as video rendering, game development, and scientific simulations. GPUs are particularly useful for tasks that require high performance, such as rendering high-resolution images, simulating complex systems

Q: Why is the sky blue?
A: The sky is blue because it reflects sunlight, which contains all colors of the visible spectrum. The blue color is due to the scattering of sunlight by tiny particles in the Earth's atmosphere, particularly water vapor and nitrogen oxides. This scattering causes the blue light to be scattered more than other colors, resulting in the characteristic blue

Q: What is a hash table?
A: A hash table is a data structure that uses a hash function to store and retrieve data. It works b

In [6]:
# K/V cache on the GPU, with attention in paged-kv-cuda's kernel.
if HAS_GPU:
    !cd paged-kv-cache && cargo run --release -q -p paged-kv-model --features cuda --example qwen -- {MODEL_DIR} {MAX_NEW_TOKENS}
else:
    print("No GPU on this runtime, so this run is skipped.")

K/V cache on cuda:0 (48 MiB pool)

Q: What does a GPU do?
A: A GPU (Graphics Processing Unit) is a specialized computer component designed to perform complex mathematical calculations and image processing tasks, enabling it to handle graphics-intensive applications such as video rendering, game development, and scientific simulations. GPUs are particularly useful for tasks that require high performance, such as rendering high-resolution images, simulating complex systems

Q: Why is the sky blue?
A: The sky is blue because it reflects sunlight, which contains all colors of the visible spectrum. The blue color is due to the scattering of sunlight by tiny particles in the Earth's atmosphere, particularly water vapor and nitrogen oxides. This scattering causes the blue light to be scattered more than other colors, resulting in the characteristic blue

Q: What is a hash table?
A: A hash table is a data structure that uses a hash function to store and retrieve data. It works by mapping keys 